In [1]:
import pyautogui as pyg # 後續可用於畫面自動化

# 若前面的 cell 尚未設定這些值，就在此輸入左上角中心座標與間距。
if 'basex' not in globals():
    basex = int(input('左上角寶石中心 X 座標 basex：'))
if 'basey' not in globals():
    basey = int(input('左上角寶石中心 Y 座標 basey：'))
if 'gap' not in globals():
    gap = int(input('寶石中心間距 gap：'))

# 逐列、逐欄計算並列印 8×8 共 64 顆寶石的中心點。
gem_centers = []
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        center = (x, y)
        gem_centers.append(center)
        print(f'第 {row + 1} 列第 {col + 1} 欄：{center}')


第 1 列第 1 欄：(489, 440)
第 1 列第 2 欄：(529, 440)
第 1 列第 3 欄：(569, 440)
第 1 列第 4 欄：(609, 440)
第 1 列第 5 欄：(649, 440)
第 1 列第 6 欄：(689, 440)
第 1 列第 7 欄：(729, 440)
第 1 列第 8 欄：(769, 440)
第 2 列第 1 欄：(489, 480)
第 2 列第 2 欄：(529, 480)
第 2 列第 3 欄：(569, 480)
第 2 列第 4 欄：(609, 480)
第 2 列第 5 欄：(649, 480)
第 2 列第 6 欄：(689, 480)
第 2 列第 7 欄：(729, 480)
第 2 列第 8 欄：(769, 480)
第 3 列第 1 欄：(489, 520)
第 3 列第 2 欄：(529, 520)
第 3 列第 3 欄：(569, 520)
第 3 列第 4 欄：(609, 520)
第 3 列第 5 欄：(649, 520)
第 3 列第 6 欄：(689, 520)
第 3 列第 7 欄：(729, 520)
第 3 列第 8 欄：(769, 520)
第 4 列第 1 欄：(489, 560)
第 4 列第 2 欄：(529, 560)
第 4 列第 3 欄：(569, 560)
第 4 列第 4 欄：(609, 560)
第 4 列第 5 欄：(649, 560)
第 4 列第 6 欄：(689, 560)
第 4 列第 7 欄：(729, 560)
第 4 列第 8 欄：(769, 560)
第 5 列第 1 欄：(489, 600)
第 5 列第 2 欄：(529, 600)
第 5 列第 3 欄：(569, 600)
第 5 列第 4 欄：(609, 600)
第 5 列第 5 欄：(649, 600)
第 5 列第 6 欄：(689, 600)
第 5 列第 7 欄：(729, 600)
第 5 列第 8 欄：(769, 600)
第 6 列第 1 欄：(489, 640)
第 6 列第 2 欄：(529, 640)
第 6 列第 3 欄：(569, 640)
第 6 列第 4 欄：(609, 640)
第 6 列第 5 欄：(649, 640)
第 6 列第 6 欄

In [2]:
def get_gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石的中心座標；r、c 使用 0 到 7 的索引。"""
    if not (0 <= r < 8 and 0 <= c < 8):
        raise ValueError('r 和 c 必須介於 0 到 7')

    x = basex + c * gap
    y = basey + r * gap
    return (x, y)


In [3]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

(489, 440)
(529, 440)
(489, 480)


In [4]:
im=pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0,i)
    pyg.click(x,y,duration=1)

In [5]:
def get_gem_color(r, c, im):
    """回傳截圖 im 中第 r 列、第 c 欄寶石中心點的顏色。"""
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))


In [6]:
def can_move_right(r, c, im):
    """判斷第 r 列、第 c 欄寶石與右側寶石交換後是否會形成三連線。"""
    if not (0 <= r < 8 and 0 <= c < 7):
        return False

    board = [[get_gem_color(row, col, im) for col in range(8)] for row in range(8)]

    def find_matches(gems):
        matches = set()

        # 找出橫向連續三顆以上的同色寶石。
        for row in range(8):
            col = 0
            while col < 8:
                end = col + 1
                while end < 8 and gems[row][end] == gems[row][col]:
                    end += 1
                if end - col >= 3:
                    cells = tuple((row, current_col) for current_col in range(col, end))
                    matches.add((gems[row][col], cells))
                col = end

        # 找出縱向連續三顆以上的同色寶石。
        for col in range(8):
            row = 0
            while row < 8:
                end = row + 1
                while end < 8 and gems[end][col] == gems[row][col]:
                    end += 1
                if end - row >= 3:
                    cells = tuple((current_row, col) for current_row in range(row, end))
                    matches.add((gems[row][col], cells))
                row = end

        return matches

    matches_before = find_matches(board)
    board[r][c], board[r][c + 1] = board[r][c + 1], board[r][c]
    matches_after = find_matches(board)
    return bool(matches_after - matches_before)


In [7]:
def _can_swap_and_match(r, c, target_r, target_c, im):
    """交換兩顆寶石，判斷是否因此形成新的三連線。"""
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= target_r < 8 and 0 <= target_c < 8):
        return False

    board = [[get_gem_color(row, col, im) for col in range(8)] for row in range(8)]

    def find_matches(gems):
        matches = set()

        for row in range(8):
            col = 0
            while col < 8:
                end = col + 1
                while end < 8 and gems[row][end] == gems[row][col]:
                    end += 1
                if end - col >= 3:
                    cells = tuple((row, current_col) for current_col in range(col, end))
                    matches.add((gems[row][col], cells))
                col = end

        for col in range(8):
            row = 0
            while row < 8:
                end = row + 1
                while end < 8 and gems[end][col] == gems[row][col]:
                    end += 1
                if end - row >= 3:
                    cells = tuple((current_row, col) for current_row in range(row, end))
                    matches.add((gems[row][col], cells))
                row = end

        return matches

    matches_before = find_matches(board)
    board[r][c], board[target_r][target_c] = board[target_r][target_c], board[r][c]
    matches_after = find_matches(board)
    return bool(matches_after - matches_before)

def can_move_left(r, c, im):
    """判斷第 r 列、第 c 欄寶石是否可以向左移動形成三連線。"""
    if not (0 <= r < 8 and 0 < c < 8):
        return False
    return _can_swap_and_match(r, c, r, c - 1, im)

def can_move_up(r, c, im):
    """判斷第 r 列、第 c 欄寶石是否可以向上移動形成三連線。"""
    if not (0 < r < 8 and 0 <= c < 8):
        return False
    return _can_swap_and_match(r, c, r - 1, c, im)

def can_move_down(r, c, im):
    """判斷第 r 列、第 c 欄寶石是否可以向下移動形成三連線。"""
    if not (0 <= r < 7 and 0 <= c < 8):
        return False
    return _can_swap_and_match(r, c, r + 1, c, im)


In [8]:
import time

start_time = time.time()
im = pyg.screenshot()
while True:
    if time.time() - start_time > 60:
        break

    moved = False

    for r in range(8):
        for c in range(8):
            if time.time() - start_time > 60:
                break

            directions = (
                (can_move_right, 0, 1),
                (can_move_left, 0, -1),
                (can_move_up, -1, 0),
                (can_move_down, 1, 0),
            )
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2, button='left')
                    pyg.sleep(0.3)
                    im = pyg.screenshot()
                    moved = True
                    break

            if moved:
                break
        if moved or time.time() - start_time > 60:
            break

    if not moved:
        pyg.sleep(0.1)


FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.